# Solar filaments: batch Dice and additional training

This notebook reproduces the released all-data model's predictions, or runs its two-stage
training procedure. A grayscale H-alpha image is mapped to filament foreground probabilities;
native-resolution connected components produce disjoint instance masks. The score is pooled
per-annotator Panoptic Quality with strict IoU > 0.5.

Stage one fits the original U-Net recipe for 6,000 updates. Stage two uses batch Dice plus BCE
for 3,000 additional updates at a smaller learning rate. The comparison tests both changes
together; it does not isolate the loss from the extra training. Only approved competition
training labels are used. The full public MAGFiLO archive contains test labels and is prohibited.

The recipe has a separately frozen three-fold subsequent-development comparison. All 707
approved images train the released model, so it has no independent internal holdout score.
The old baseline's 0.34625 holdout PQ does not apply to this model. Shared inner calibration and
earlier work on the corpus limit independence; see the exact comparison record below.

The default run loads weights and performs fresh CPU inference. Set `RUN_TRAINING=True` to
fine-tune the released, approved stage-one checkpoint. Also set `RETRAIN_PARENT=True` in the
training cell to refit stage one from scratch. Inference replay alone is not retraining.
No competition submission occurs automatically.


In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import urllib.request
import zipfile

SOURCE_COMMIT = 'f75664a6ff7a4cb7c6e78a8b1b795ed95928ac66'
EXPECTED_SOURCE = {'solarseg/__init__.py': '32ca1f85bbbc5f820eac7e237cd01ec16027cc5b3e15028d2af265483e0a9b5b', 'solarseg/__main__.py': 'f887e0af26ca51dc3e89f873e775ffdf4f55426a3d8d25204c4b065a8f312979', 'solarseg/data.py': '46152a04d6eafbbaeaa234b0bb461450c0b59324a68fb05d029d71cab3a1313b', 'solarseg/engine.py': '9358ebc99df66eb669f32eb83ab8118502f799eaddd9e59d3823b66e6d85e0c5', 'solarseg/gates.py': '4081730062166a57be537f2c4bfebf936b305449a1414356b205b0b1cba753f3', 'solarseg/metrics.py': '9a2d5e5c1bbc7a89250311d1d45af4cda8cfaac8c7eff7cddf5746f69b015b6b', 'solarseg/model.py': '37fe7b0e2edc4997a8221fca12f8eec2b6f5314a665ef89a354a73e9ced1238a', 'solarseg/postprocess.py': '9b779c5079ed33cff64e521612726ee67a5222506aa61738c2bd7d8c861cb9ce', 'solarseg/provenance.py': '2de76b5d199270f50427787e046a1833293697146064b927c342a8d096fa8ad1', 'solarseg/submission.py': '8dc9b08bf2ef1ce473fc28ccc3f49f2fa3e526988cf102d0c4b20d70072ad489', 'requirements.txt': '12fac5b80df1c561b5cd85e4ba29f580ed13ebffd6e024bb3ae26a0339a567d2'}
RUN_TRAINING = False
RUN_HOLDOUT = False
REPO = "srivatsav-kannan/solar-seg"

ROOT = Path.cwd()
if not (ROOT / "solarseg").is_dir():
    if (ROOT.parent / "solarseg").is_dir():
        ROOT = ROOT.parent
    else:
        url = f"https://github.com/{REPO}/archive/{SOURCE_COMMIT}.zip"
        payload = urllib.request.urlopen(url, timeout=120).read()
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                target = (ROOT / member.filename).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError("Unsafe source archive path")
            archive.extractall(ROOT)
        ROOT = ROOT / f"solar-seg-{SOURCE_COMMIT}"
for name, expected in EXPECTED_SOURCE.items():
    actual = hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
    assert actual == expected, f"Source fingerprint differs: {name}"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
assert sys.version_info >= (3, 12), "Use Python 3.12 or newer with the pinned dependencies"
print("Source revision:", SOURCE_COMMIT)

COMMON_INIT = '\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport urllib.request\nimport zipfile\n\nSOURCE_COMMIT = \'f75664a6ff7a4cb7c6e78a8b1b795ed95928ac66\'\nEXPECTED_SOURCE = {\'solarseg/__init__.py\': \'32ca1f85bbbc5f820eac7e237cd01ec16027cc5b3e15028d2af265483e0a9b5b\', \'solarseg/__main__.py\': \'f887e0af26ca51dc3e89f873e775ffdf4f55426a3d8d25204c4b065a8f312979\', \'solarseg/data.py\': \'46152a04d6eafbbaeaa234b0bb461450c0b59324a68fb05d029d71cab3a1313b\', \'solarseg/engine.py\': \'9358ebc99df66eb669f32eb83ab8118502f799eaddd9e59d3823b66e6d85e0c5\', \'solarseg/gates.py\': \'4081730062166a57be537f2c4bfebf936b305449a1414356b205b0b1cba753f3\', \'solarseg/metrics.py\': \'9a2d5e5c1bbc7a89250311d1d45af4cda8cfaac8c7eff7cddf5746f69b015b6b\', \'solarseg/model.py\': \'37fe7b0e2edc4997a8221fca12f8eec2b6f5314a665ef89a354a73e9ced1238a\', \'solarseg/postprocess.py\': \'9b779c5079ed33cff64e521612726ee67a5222506aa61738c2bd7d8c861cb9ce\', \'solarseg/provenance.py\': \'2de76b5d199270f50427787e046a1833293697146064b927c342a8d096fa8ad1\', \'solarseg/submission.py\': \'8dc9b08bf2ef1ce473fc28ccc3f49f2fa3e526988cf102d0c4b20d70072ad489\', \'requirements.txt\': \'12fac5b80df1c561b5cd85e4ba29f580ed13ebffd6e024bb3ae26a0339a567d2\'}\nRUN_TRAINING = False\nRUN_HOLDOUT = False\nREPO = "srivatsav-kannan/solar-seg"\n\nROOT = Path.cwd()\nif not (ROOT / "solarseg").is_dir():\n    if (ROOT.parent / "solarseg").is_dir():\n        ROOT = ROOT.parent\n    else:\n        url = f"https://github.com/{REPO}/archive/{SOURCE_COMMIT}.zip"\n        payload = urllib.request.urlopen(url, timeout=120).read()\n        with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n            for member in archive.infolist():\n                target = (ROOT / member.filename).resolve()\n                if not target.is_relative_to(ROOT.resolve()):\n                    raise ValueError("Unsafe source archive path")\n            archive.extractall(ROOT)\n        ROOT = ROOT / f"solar-seg-{SOURCE_COMMIT}"\nfor name, expected in EXPECTED_SOURCE.items():\n    actual = hashlib.sha256((ROOT / name).read_bytes()).hexdigest()\n    assert actual == expected, f"Source fingerprint differs: {name}"\nos.chdir(ROOT)\nsys.path.insert(0, str(ROOT))\nassert sys.version_info >= (3, 12), "Use Python 3.12 or newer with the pinned dependencies"\nprint("Source revision:", SOURCE_COMMIT)\n'

# Kaggle may preload NumPy before this cell. Never upgrade its live kernel.
# Each %%solarseg cell below runs in one persistent, clean child kernel there.
from IPython.core.magic import register_cell_magic

ISOLATED_KERNEL = Path("/kaggle").exists() or os.environ.get("SOLARSEG_ISOLATE") == "1"
if ISOLATED_KERNEL:
    import atexit
    from jupyter_client import KernelManager

    isolated_python = os.environ.get("SOLARSEG_ISOLATED_PYTHON")
    if not isolated_python:
        env_dir = Path("/tmp/solarseg-pinned-env")
        # Kaggle's Debian Python omits ensurepip; host pip can manage a pip-less venv.
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(env_dir)], check=True)
        isolated_python = str(env_dir / "bin/python")
        pip_command = [sys.executable, "-m", "pip", "--python", isolated_python]
        # CPU build avoids downloading unused CUDA libraries for this CPU replay.
        subprocess.run([*pip_command, "install", "--quiet", "--no-cache-dir",
                        "torch==2.14.0", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
        subprocess.run([*pip_command, "install", "--quiet", "--no-cache-dir",
                        "-r", str(ROOT / "requirements.txt")], check=True)
    kernel_manager = KernelManager(kernel_name="python3")
    kernel_manager.kernel_spec.argv = [isolated_python, "-m", "ipykernel_launcher", "-f", "{connection_file}"]
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env.pop("PYTHONHOME", None)
    kernel_manager.start_kernel(cwd=str(ROOT), env=child_env)
    kernel_client = kernel_manager.blocking_client()
    kernel_client.start_channels()
    kernel_client.wait_for_ready(timeout=120)
    atexit.register(lambda: kernel_manager.shutdown_kernel(now=True))

    def execute_isolated(source):
        reply = kernel_client.execute_interactive(source, timeout=7200, allow_stdin=False)
        if reply["content"]["status"] != "ok":
            raise RuntimeError("Isolated cell failed: " + str(reply["content"]))

    execute_isolated(COMMON_INIT + f"\nRUN_TRAINING={RUN_TRAINING!r}\nRUN_HOLDOUT={RUN_HOLDOUT!r}")

@register_cell_magic
def solarseg(line, cell):
    if ISOLATED_KERNEL:
        execute_isolated(cell)
    else:
        result = get_ipython().run_cell(cell)
        result.raise_error()

In [ ]:
%%solarseg
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from solarseg.data import CompetitionData, make_manifest, read_image, sha256
from solarseg.engine import Predictor, train, predict_cache, calibrate, evaluate_and_save
from solarseg.metrics import aggregate, counts
from solarseg.postprocess import encode, decode, instances
from solarseg.provenance import environment, source_hashes, utc_now
from solarseg.submission import write_submission

DATA_ROOT = Path(os.environ.get("SOLARSEG_DATA", "/kaggle/input" if Path("/kaggle/input").exists() else "data/raw"))
WORK = Path("/kaggle/working/solarseg-batch-run") if Path("/kaggle").exists() else ROOT / "artifacts/notebook-batch-run"
WORK.mkdir(parents=True, exist_ok=True)
RELEASE_TAG = 'batch-dice-v0.3'
MODEL_SHA256 = '89ec54f951cfd3ec63f65d36d0368d308b8d3a58def2ba814dc5432e80d84540'
PARAMS = {'threshold': 0.6, 'min_area': 400, 'closing': 3}
TILE = 0
TTA = True
CHECKPOINT = WORK / "model.pt"
local_model = ROOT / 'artifacts/batch-dice-full-v1/model.pt'
if not CHECKPOINT.exists():
    if local_model.exists():
        shutil.copy2(local_model, CHECKPOINT)
    else:
        url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/model.pt"
        urllib.request.urlretrieve(url, CHECKPOINT)
assert sha256(CHECKPOINT) == MODEL_SHA256, "Checkpoint checksum mismatch"
print(json.dumps(environment(), indent=2))
print("Artifact directory:", WORK)
assert not RUN_HOLDOUT, "The all-data refit has no independent holdout"
from research.train import train as train_refiner


## 1. Audit inputs and freeze physical-image groups

Download with `python scripts/download_data.py` after accepting the Kaggle rules, or attach
the competition data in Kaggle. An annotation ID is not a physical image ID. All annotators,
crops, and augmentations of one observation follow the same fold. The audit uses 27-day
blocks, exact decoded-pixel hashes, and a three-day embargo on optimization data.

The original split is reproduced for provenance. A separate manifest then assigns all 707 approved observations to final training.

In [ ]:
%%solarseg
data = CompetitionData(DATA_ROOT)
audit = make_manifest(data, WORK / "manifests")
assert audit["train_annotation_sha256"] == '5da9e92b5a1a1947fd5d57adb6688269625c48ec1ef884daf2a01618c9ed54a1'
assert audit["split_manifest_sha256"] == 'ebec49b919b111f0591f1740cc10a470c127371203e7522c683c9ec5a34283b8'
manifest = pd.read_csv(WORK / "manifests/train_manifest.csv")
display(pd.Series(audit["roles"], name="Physical images").to_frame())
print("Train images / annotator records / instances / test images:",
      audit["physical_train"], audit["annotator_images"], audit["annotations"], audit["test"])
full_manifest = manifest.copy()
full_manifest["role"] = "train"
full_manifest_path = WORK / "full-training-manifest.csv"
full_manifest.to_csv(full_manifest_path, index=False)
assert len(full_manifest) == 707 and set(full_manifest.stem) == set(data.by_stem)
assert sha256(full_manifest_path) == '531fc072410a04a77bb15310d4ee3a8ccadfeb85bfa1e0a2d233d5532e2b07b5'
print("Final training observations:", len(full_manifest))

## 2. Inspect input and annotator-aware target

The foreground target averages separate annotator unions. The score compares predictions
with each annotator separately. It never evaluates against this consensus target.
The deterministic example belonged to the original calibration split and is now part of final training; this illustration is not independent evaluation.

In [ ]:
%%solarseg
stem = manifest.loc[manifest.role == "calibration", "stem"].iloc[0]
image = read_image(data.image_path(stem), 1024)
target = data.soft_target(stem, 1024)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"H-alpha: {stem}")
axes[1].imshow(target, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Mean of annotator foreground unions")
for ax in axes: ax.axis("off")
plt.show()

## 3. Verify metric and serialization contracts

PQ = sum of matched IoUs / (TP + 0.5 FP + 0.5 FN). An IoU of exactly 0.5 is not a match.
COCO uses compressed counts with column-major ordering, not ordinary Kaggle run-length pairs.
The full repository tests also compare our evaluator with the downloaded official notebook.

In [ ]:
%%solarseg
mask = np.zeros((23, 31), dtype=np.uint8)
mask[3:11, 14:25] = 1
np.testing.assert_array_equal(mask, decode(encode(mask)))
assert counts(np.array([[0.5]]))["tp"] == 0
assert aggregate([counts(np.array([[1.0]])), counts(np.zeros((3, 0)))])["pq"] == 0.4
print("Asymmetric COCO roundtrip and pooled PQ contracts passed")

## 4. Train from approved labels (opt-in)

The architecture is the same width-24 U-Net. Stage one uses BCE plus per-image Dice for
6,000 updates at initial learning rate 0.001. Stage two uses BCE plus Dice aggregated across
the batch for 3,000 updates at initial learning rate 0.0002. Both use 1024-pixel disks,
384-pixel crops, batch size six, AdamW, cosine decay, the same augmentations, and seed 2026.

Batch Dice changes how empty crops contribute to the loss. Training code lives in reusable
modules. Temporary Kaggle training caches stay outside published outputs. Saved configurations
record the actual numerical environment; CUDA training and local MPS training are not assumed
bitwise equivalent. The released inference checkpoint has an exact checksum.


In [ ]:
%%solarseg
RETRAIN_PARENT = False
training_root = Path("/tmp/solarseg-batch-training") if Path("/kaggle").exists() else WORK / "training"
parent_run = training_root / "parent"
fine_run = training_root / "refined"
if RUN_TRAINING:
    training_root.mkdir(parents=True, exist_ok=True)
    if RETRAIN_PARENT:
        train(DATA_ROOT, full_manifest_path, parent_run,
              steps=6000, size=1024, crop=384, batch_size=6, width=24, seed=2026)
        parent_checkpoint = parent_run / "model.pt"
    else:
        parent_checkpoint = training_root / "released-parent.pt"
        if not parent_checkpoint.exists():
            urllib.request.urlretrieve(
                f"https://github.com/{REPO}/releases/download/full-refit-v0.2/model.pt",
                parent_checkpoint)
        assert sha256(parent_checkpoint) == '4f96bc349e6e6c7bd22fa6921071eb4ad5a3b90c88737ba4c4c02e9a1e246a08'
    train_refiner(DATA_ROOT, full_manifest_path, fine_run,
                  steps=3000, size=1024, crop=384, batch_size=6, width=24,
                  depth=3, seed=2026, learning_rate=0.0002,
                  initialize=parent_checkpoint, loss_mode="batch")
    CHECKPOINT = fine_run / "model.pt"
    if Path("/kaggle").exists():
        published = WORK / "retrained"
        published.mkdir(exist_ok=False)
        for name in ["model.pt", "config.json", "history.json"]:
            shutil.copy2(fine_run / name, published / name)
        CHECKPOINT = published / "model.pt"
print("Training rerun:", RUN_TRAINING, "Parent refit:", RUN_TRAINING and RETRAIN_PARENT)


## 5. Apply the frozen reconstruction settings

Four-flip averaging, threshold 0.6, minimum area 400 native pixels, and closing radius 3
were fixed from development calibration before the outer comparison. The all-data fit's
training observations must not be used to retune them. Restore probabilities to 2048 x 2048
before thresholding. Small components are filtered; retained masks are nonempty and disjoint.


In [ ]:
%%solarseg
predictor = Predictor(CHECKPOINT, device="cpu", tta=TTA, tile=TILE)
print("Frozen reconstruction settings:", PARAMS)


## 6. Inspect the frozen paired validation evidence

Three outer folds compare separately trained baseline and fine-tuned models. Each fit's
thresholds are chosen on its own inner calibration partition, then frozen before any outer
inference. All annotators of a physical image and all crops stay together; time groups and a
three-day embargo separate partitions. Outer counts are pooled, and temporal groups are
resampled for paired uncertainty. The common inner-calibration partition limits independence.

The exact models, manifests, training commands, and evaluation code are in the repository's
`configs/batch-dice-cv-02.json`, `notebooks/batch-dice-cv-02/`, and
`research/evaluate_batch_cv.py`. The downloaded record below is historical validation evidence,
not a claim that this notebook reruns cross-validation. It does not evaluate the all-data
checkpoint on its own training images.


In [ ]:
%%solarseg
CV_SHA256 = 'c690bc0b494bb9993f48b9d1886b9c65243eeb181daf7d30923fa224e57c228c'
CV_RECORD = WORK / "cv-comparison.json"
local_cv = ROOT / "artifacts/cv-02/paired-batch-dice/comparison.json"
if not CV_RECORD.exists():
    if local_cv.exists():
        shutil.copy2(local_cv, CV_RECORD)
    else:
        urllib.request.urlretrieve(
            f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/cv-comparison.json",
            CV_RECORD)
assert sha256(CV_RECORD) == CV_SHA256
cv = json.loads(CV_RECORD.read_text())
assert cv["statistical_gates_passed"]
print(json.dumps({key: cv[key] for key in ["pq_gain", "paired_block_bootstrap_95ci", "fold_pq", "scope"]}, indent=2))


## 7. Infer every test observation and write a valid submission

Test labels are neither available nor required. Every test image is processed, including
images for which the model predicts zero instances. Zero detections go in the coverage
sidecar; the CSV contains only real nonempty masks. IDs preserve the original image stem.
No automatic leaderboard probing, dummy masks, or manual test corrections are performed.

In [ ]:
%%solarseg
started = time.monotonic()
stems = sorted(data.test_paths)
cache = WORK / "probabilities-test"
predict_cache(data, stems, predictor, cache)
validation = write_submission(stems, cache, PARAMS, WORK / "submission.csv")
assert validation["images_processed"] == audit["test"]
elapsed = time.monotonic() - started
print(json.dumps(validation, indent=2))
print(f"Inference + serialization seconds (including any cache reuse): {elapsed:.1f}")
display(pd.read_csv(WORK / "submission.csv").head())

## 8. Record evidence and submit only the checked file

Submission requires the paired CV gate, morphology review, the approved full-data training
recipe, exact checkpoint and input fingerprints, a fresh CPU notebook replay, and a valid CSV
covering all 180 test observations. Check quota immediately before uploading and retain the
server receipt. The notebook never uploads automatically. The competition's report and
authenticated final form remain separate final-entry obligations.


In [ ]:
%%solarseg
proof = {"created_at": utc_now(), "completed": True, "source_commit": SOURCE_COMMIT,
         "source_hashes": source_hashes(), "checkpoint_sha256": sha256(CHECKPOINT),
         "environment": environment(), "validation": validation, "elapsed_seconds": elapsed,
         "training_rerun": RUN_TRAINING, "holdout_rerun": False, "full_data_refit": True,
         "training_manifest_sha256": sha256(full_manifest_path),
         "cv_comparison_sha256": sha256(CV_RECORD), "recipe": "baseline then batch-Dice fine-tuning",
         "validation_scope": 'All 707 approved observations are training data. No independent internal holdout score is available for these weights. Promotion requires the separate paired CV gates and morphology review.'}
(WORK / "notebook-proof.json").write_text(json.dumps(proof, indent=2))
if Path("/kaggle").exists():
    shutil.rmtree(cache)
print("Validated CSV:", WORK / "submission.csv")
